In [1]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re


In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.max_colwidth', None)


In [3]:
df = pd.read_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')

In [4]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method', 'CMO_before', 'CMO_after', 'MMO_before',
       'MMO

### 데이터 전처리

In [5]:
## NaN 처리 

df = df.replace('', np.nan)
df = df.replace('-', np.nan)
df = df.replace('- -', np.nan)
df = df.replace('n/s', np.nan)

df['약_medication_type'] = df['약_medication_type'].replace('없음',np.nan)
df['약_medication_type'] = df['약_medication_type'].replace('약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('약물 종류','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용함','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다복용','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('저녁약','약물 종류 미상')
df['약_medication_type'] = df['약_medication_type'].replace('다양한 약물','약물 종류 미상')

df['Noise_Code'] = df['Noise_Code'].apply(lambda x : "No-Noise" if x == 0 else "Click" if x == 1 else "Popping" if x == 2 else "Crepitus" if x == 3 else "Unknown")

text_cols = [
    'CC_location','CC_pain_type','CC_painUncomp_desc_jaw','CC_disable_desc_jaw','CC_muscle_joint_desc_stress',
    'CC_dentalHistory_desc','CC_clinic_history_desc','CC_factor_habbit','CC_treat_plan',
    '약_medication_type','약_compliance','장치_device_type','습관_habit_type','습관_awareness'
    ]
numeric_cols = [
    'CC_duration','CC_severity', 'CC_vas', 'CMO_before','CMO_after','MMO_before','MMO_after','Midline_Shift_Amount','CRCO_Amount','Next_Visit_Days'
    ,'Rt_before','Rt_after','Lt_before','Lt_after','Tongue_ridging_Intensity','장치_duration','찜질_duration','마사지, 스트레칭_duration','약_duration'
    ,'M.pal_Pain_Intensity','Cap.pal_Pain_Intensity','Noise_Intensity','Occlusion_lt_Intensity','Occlusion_rt_Intensity', 'oj','ob'
    ]
category_cols = [
    '장치_usage_pattern','장치_compliance', '습관_frequency', '습관_improvement','약_frequency',
    '찜질_status','찜질_frequency', '마사지, 스트레칭_frequency','마사지, 스트레칭_method' ,'deviation_pattern_type','deviation_direction',
    'Cap.pal_Pain_Direction','Cap.pal_Pain_Situation','M.pal_Pain_Direction','M.pal_Pain_Situation','Noise_Code','Noise_Direction',
    'Noise_Situation','Occlusion_lt_number','Occlusion_rt_number','Midline_Shift_Jaw','Midline_Shift_Direction_x','Midline_Shift_Direction_y',
    'CRCO_Direction_x','CRCO_Direction_y','Cap.pal_Pain_Intensity','deviation_intensity',
    '마사지, 스트레칭_frequency','마사지, 스트레칭_type','찜질_method'
    ]


In [6]:
## Vas 추출 

def get_vas_sentence(text):
    if isinstance(text, str) and 'vas' in text.lower():
        vas_index = text.lower().find('vas')
        vas_after = text[vas_index:]
        vas_after = vas_after.replace(" ", "")
        vas_after = vas_after.lower()
        return vas_after

def get_vas_value(text):    
    pattern = re.compile(
        # r"(?i)VAS(?:\s*(?:,|~|->)?\s*(\d+(?:\.\d+)?))+"
        r"(?i)VAS(?:\s*(?:[,\.\~\/]|->)\s*)?"      # 'VAS' 뒤에 선택적으로 구분자가 올 수 있음
        r"(\d+(?:\.\d+)?)"                         # 첫 번째 숫자 (정수 또는 소수)
        r"(?:\s*(?:[,\.\~\/]|->)\s*(\d+(?:\.\d+)?))*"  # 이후 반복되는 구분자와 숫자; 마지막 반복의 캡처 그룹에 최종 숫자가 남음
        )
    if text:
        match = pattern.search(text)
        if match:
            # print(match.group(1))
            return match.group(2) if match.group(2) is not None else match.group(1)
            # return match.group(1)
    else:
        return None


df['CC_vas_sentence'] = df['CC'].apply(get_vas_sentence)
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('-->','->')
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('--->','->')
df['CC_vas_sentence'] =  df.CC_vas_sentence.replace('---->','->')

df['CC_vas_2'] = df['CC_vas_sentence'].apply(get_vas_value)

df.drop(columns=['CC_vas_sentence'], inplace=True)

In [7]:
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
    '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method', 'CMO_before', 'CMO_after', 'MMO_before',
       'MMO_after', 'deviation_pattern_type', 'deviation_direction',
       'deviation_intensity', 'Cap.pal_Pain_Intensity',
       'Cap.pal_Pain_Direction', 'Cap.pal_Pain_Situation',
       'M.pal_Pain_Intensity', 'M.pal_Pain_Direction', 'M.pal_Pain_Situation',
       'Noise_Code', 'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days', 'CC_vas_2'
    ]]


In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28162 entries, 0 to 28161
Data columns (total 80 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   환자번호                         28162 non-null  object        
 1   날짜                           28162 non-null  datetime64[ns]
 2   CC                           28161 non-null  object        
 3   약                            5897 non-null   object        
 4   장치                           11187 non-null  object        
 5   습관                           16035 non-null  object        
 6   찜질                           16917 non-null  object        
 7   마사지, 스트레칭                    5935 non-null   object        
 8   PI                           17480 non-null  object        
 9   치료계획                         20489 non-null  object        
 10  End feel                     25630 non-null  object        
 11  CC_location                  26760 non-nu

In [9]:
def structure_patient_data(row):
    structured_text = ""
    for column in text_columns:
        if pd.notna(row[column]) and row[column]:
            structured_text += f"{column}: {row[column]}\n"
    return structured_text

In [11]:
df[numeric_cols].head()

,CC_duration,CC_severity,CC_vas,CMO_before,CMO_after,MMO_before,MMO_after,Midline_Shift_Amount,CRCO_Amount,Next_Visit_Days,Rt_before,Rt_after,Lt_before,Lt_after,Tongue_ridging_Intensity,장치_duration,찜질_duration,"마사지, 스트레칭_duration",약_duration,M.pal_Pain_Intensity,Cap.pal_Pain_Intensity,Noise_Intensity,Occlusion_lt_Intensity,Occlusion_rt_Intensity,oj,ob
0,30년,NaN,NaN,34,NaN,46,NaN,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,NaN,NaN,NaN,NaN,-1,-1,0,0,0,2.0,2.0
1,NaN,NaN,2.0,38,NaN,46,NaN,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,NaN,10.0,NaN,NaN,2,-1,0,0,0,2.0,2.0
2,3일,NaN,NaN,38,NaN,46,NaN,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,NaN,NaN,NaN,NaN,2,-1,0,0,0,2.0,2.0
3,NaN,4.0,3.0,40,NaN,48,NaN,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,3일,NaN,NaN,NaN,-1,-1,0,0,0,2.0,2.0
4,NaN,NaN,5.0,48,NaN,48,NaN,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,NaN,10.0,NaN,NaN,-1,-1,0,0,0,2.0,2.0


In [12]:
test1 = col1.columns.tolist()
test = list(set(text_cols + numeric_cols + category_cols))



NameError: name 'col1' is not defined

In [11]:
col1 = df.iloc[:,28:]
col0 = df.iloc[:,0:2]
combined_df = pd.concat([col0, col1], axis=1)
combined_df


,환자번호,날짜,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,...,Midline_Shift_Amount,CRCO_Direction_x,CRCO_Direction_y,CRCO_Amount,Tongue_ridging_Intensity,Rt_before,Rt_after,Lt_before,Lt_after,Next_Visit_Days
0,2301-01,2023-01-17,오른쪽 턱,통증,오른쪽 턱의 통증,오른쪽 턱의 제한된 개구,스트레스로 인한 턱 근육의 긴장,"교정 치료, 보톡스, 물리치료",왼쪽통증과 입벌림이 힘들어서 서울대병원 30년 전 장치도 했었어요. 장치는 두고 왔...,이악무는습관없어요. 이갈이 어렸을 때 만 잠은 잘자요. 스트레스 딱히 없어요. (현...,...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
1,2301-01,2023-02-01,오른쪽 턱,불편감,"턱이 불편했고, 앞에부분이 아팠음",턱벌어지는 것이 비슷함,스트레스로 인한 턱 근육의 긴장,"교정 치료, 보톡스, 물리치료","턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
2,2301-01,2023-02-17,유치 안쪽 잇몸,통증,턱 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
3,2301-01,2023-03-21,오른쪽 아랫턱,통증,조금 아플때도 있고 불편할때도 있는거,뻐쩍찌근한 느낌,스트레스로 인한 긴장,"교정 치료, 보톡스, 물리치료","턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
4,2301-01,2023-04-21,오른쪽,통증,턱의 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,레진치료,30년된 장치,치아끼리 안닿게 노력,...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28157,2405-86,2024-05-21,구강내과,통증,턱 통증,턱 관절의 비정상적인 움직임,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","입 벌어지는 것도 똑같고, 소리나는 것도 비슷해요",...,2.0,None,None,NaN,1,1.2,1.65,1.2,1.65,14.0
28158,2405-88,2024-05-22,구강,소리,,,스트레스로 인한 턱 근육의 긴장,물리치료,"장치 ck, x-ray구강내과#6증상: 소리만 있어요 , 어긋나거나 모래걸리는 소리...","환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,0,0.96,1.34,0.96,1.34,NaN
28159,2405-89,2024-05-21,오른쪽 턱,"모래갈리는 소리, 덜그덕거리는 느낌, 어긋나는 순간 통증","턱 통증, 모래갈리는 소리, 덜그덕거리는 느낌",어긋나는 순간 통증,,물리치료,장치 ck구강내과#7증상: 피곤할 때는 두통 있어요 . 오른쪽 턱 모래갈...,"환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",...,1.5,None,None,NaN,1,0.84,1.21,0.84,1.21,NaN
28160,2405-96,2024-05-29,오른쪽 턱,"모래갈리는 소리, 덜그덕거리는 느낌",통증은 거의 없음,없음,스트레스로 인한 턱 근육 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,0.73,1.14,0.73,1.14,NaN
